# Moment Retrieval - Group 6

This notebook tackles the moment-retrieval part of Assignment 2 step by step.

Goal: use the events detected by the VLM as text queries, run two pretrained moment-retrieval models, predict timestamps, compare feature settings, and save the results for IoU evaluation.

## Step-by-Step Plan

| Step | Goal | Check |
|------|------|-------|
| **6.1** | Load detected events from Step 3 | We should see clean event lists for videos 21-24 |
| **6.2** | Generate 3-5 query phrasings per event | Each event should have multiple readable query variants |
| **6.3** | Prepare videos for moment retrieval | Long videos should be split into overlapping chunks |
| **6.4** | Install/load Lighthouse and pretrained checkpoints | Moment-DETR and CG-DETR should be available |
| **6.5** | Run model 1: Moment-DETR | Each query should return timestamps and confidence scores |
| **6.6** | Run model 2: CG-DETR or QD-DETR | Same queries, comparable results |
| **6.7** | Run feature experiment | Compare CLIP against CLIP + SlowFast if possible |
| **6.8** | Post-process and save predictions | Output JSON/CSV ready for IoU evaluation |

We start with **Step 6.1** only. After checking it, we move to Step 6.2.

## Step 6.1 - Load Detected Events

Moment-retrieval models take a text query and a video as input. Our text queries come from the event-only VLM outputs created in Step 3.

The saved files are expected here:

```text
outputs/event_only_results_video_21.json
outputs/event_only_results_video_22.json
outputs/event_only_results_video_23.json
outputs/event_only_results_video_24.json
```

This step loads those files and parses the raw numbered VLM text into this structure:

```python
{
    "video_21": ["event text", "event text", ...],
    "video_22": ["event text", "event text", ...],
}
```

Success check: each video should print a non-empty list of detected events.

In [1]:
from pathlib import Path
import json
import re

EVENT_RESULTS_DIR = Path("outputs")
EXPECTED_VIDEOS = ["video_21", "video_22", "video_23", "video_24"]


def clean_event_text(text):
    """Remove numbering/prefixes and normalize whitespace."""
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"^[Ss]al[ei]e?nt\s+event\s+\d+\s*:\s*", "", text)
    text = re.sub(r"^\d+[\.)\-]\s*", "", text).strip()
    return text.rstrip()


def parse_numbered_events(raw_text):
    """Parse numbered VLM output while preserving multiline event descriptions."""
    events = []
    current = []

    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue

        starts_new_event = (
            re.match(r"^\d+[\.)\-]\s+", line)
            or re.match(r"^[Ss]al[ei]e?nt\s+event\s+\d+\s*:", line)
        )

        if starts_new_event:
            if current:
                event = clean_event_text(" ".join(current))
                if event:
                    events.append(event)
            current = [line]
        elif current:
            current.append(line)

    if current:
        event = clean_event_text(" ".join(current))
        if event:
            events.append(event)

    return events


def load_detected_events(results_dir=EVENT_RESULTS_DIR):
    """Load all saved event-only VLM outputs from the outputs folder."""
    detected_events = {}

    for video_name in EXPECTED_VIDEOS:
        json_path = results_dir / f"event_only_results_{video_name}.json"
        if not json_path.exists():
            print(f"Missing: {json_path}")
            detected_events[video_name] = []
            continue

        with open(json_path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        raw_output = payload.get(video_name, {}).get("raw", "")
        detected_events[video_name] = parse_numbered_events(raw_output)

    return detected_events


detected_events = load_detected_events()

print("Detected VLM events ready for moment retrieval:")
for video_name in EXPECTED_VIDEOS:
    events = detected_events.get(video_name, [])
    print(f"\n{video_name}: {len(events)} events")
    for i, event in enumerate(events[:3], start=1):
        print(f"  {i}. {event}")
    if len(events) > 3:
        print(f"  ... {len(events) - 3} more events")

assert all(detected_events.get(video_name) for video_name in EXPECTED_VIDEOS), "At least one video has no detected events."
print("\nStep 6.1 check passed: all expected videos have detected events.")

Detected VLM events ready for moment retrieval:

video_21: 24 events
  1. A man is running down a street with buildings behind him.
  2. The same person runs up to another building's entrance.
  3. He jumps over an obstacle while still near that building.
  ... 21 more events

video_22: 6 events
  1. A group of people are gathered around a brick wall with some standing closer to it than others.
  2. One person is jumping off the top step onto another set of steps below them while everyone else watches.
  3. The man who jumped has fallen down but quickly gets back up again after landing.
  ... 3 more events

video_23: 6 events
  1. A person is walking down a staircase with their hands behind them. They are wearing light blue jeans and carrying a backpack over their shoulder. The stairs have metal railings along both sides.
  2. Another individual walks up an elevator shaft towards another set of doors at ground level.
  3. An escalator moves upwards as someone stands near it waiting to 

## Step 6.2 - Generate Query Variants

Moment-retrieval models are sensitive to wording, so we will not use only one query per event.

For each detected event from Step 6.1, this step creates 3-5 deterministic query phrasings. These variants will later be sent to Moment-DETR and CG-DETR, and we can keep the highest-scoring timestamp prediction.

Success check: every event should have at least 3 query variants.

In [ ]:
def make_query_variants(event, max_variants=5):
    """Create deterministic text-query variants for one detected event."""
    base = re.sub(r"\s+", " ", event).strip().rstrip(".")
    lower = base[:1].lower() + base[1:] if base else base

    variants = [
        base,
        f"A scene where {lower}.",
        f"The video shows {lower}.",
        f"The relevant moment is when {lower}.",
    ]

    # Add one simplified version when the VLM wording is overly specific.
    simplified = lower
    simplified = re.sub(r"\bthe same\b", "the", simplified, flags=re.IGNORECASE)
    simplified = re.sub(r"\banother individual\b", "a person", simplified, flags=re.IGNORECASE)
    simplified = re.sub(r"\banother\b", "a", simplified, flags=re.IGNORECASE)
    simplified = re.sub(r"\bindividual\b", "person", simplified, flags=re.IGNORECASE)
    simplified = re.sub(r"\s+", " ", simplified).strip()

    if simplified and simplified != lower:
        variants.append(simplified)

    # Remove duplicates while preserving order.
    unique_variants = []
    seen = set()
    for variant in variants:
        variant = re.sub(r"\s+", " ", variant).strip()
        key = variant.lower().rstrip(".")
        if variant and key not in seen:
            seen.add(key)
            unique_variants.append(variant)

    return unique_variants[:max_variants]


query_variants = {
    video_name: [
        {
            "event_id": event_id,
            "event": event,
            "queries": make_query_variants(event),
        }
        for event_id, event in enumerate(events, start=1)
    ]
    for video_name, events in detected_events.items()
}

print("Query variants generated:")
for video_name in EXPECTED_VIDEOS:
    event_queries = query_variants.get(video_name, [])
    counts = [len(item["queries"]) for item in event_queries]
    min_count = min(counts) if counts else 0
    max_count = max(counts) if counts else 0
    print(f"{video_name}: {len(event_queries)} events, {min_count}-{max_count} queries per event")

print("\nPreview: first event of each video")
for video_name in EXPECTED_VIDEOS:
    item = query_variants[video_name][0]
    print(f"\n{video_name} - event {item['event_id']}: {item['event']}")
    for i, query in enumerate(item["queries"], start=1):
        print(f"  q{i}. {query}")

assert all(
    len(item["queries"]) >= 3
    for event_queries in query_variants.values()
    for item in event_queries
), "At least one event has fewer than 3 query variants."

print("\nStep 6.2 check passed: every event has at least 3 query variants.")

## Step 6.3 - Prepare Videos for Moment Retrieval

Pretrained moment-retrieval demos often work best with short videos or clips. The Lighthouse demo notebook notes a practical limit around **150 seconds**, so longer videos should be split into overlapping chunks before retrieval.

In this step we do **not** run a model yet. We only create a chunk plan:

- get each video duration,
- keep short videos as one chunk,
- split long videos into chunks shorter than 150 seconds,
- add overlap so events near chunk boundaries are not missed,
- store everything in `video_chunk_plan` for the retrieval steps.

Success check: `video_21` should be split into several chunks; shorter videos should have fewer chunks.

In [ ]:
from pathlib import Path
import math

VIDEO_PATHS = {
    "video_21": Path("video_21.mp4"),
    "video_22": Path("video_22.mp4"),
    "video_23": Path("video_23.mp4"),
    "video_24": Path("video_24.mp4"),
}

# Fallback durations from the earlier metadata cell in Assignment2_Group6.ipynb.
# These are used only if OpenCV metadata reading is unavailable in the current kernel.
KNOWN_DURATIONS = {
    "video_21": 647.51,
    "video_22": 235.88,
    "video_23": 187.89,
    "video_24": 145.35,
}

MAX_CHUNK_SECONDS = 149.0   # stay just below the ~150s practical model/demo limit
CHUNK_OVERLAP_SECONDS = 10.0


def seconds_to_mmss(seconds):
    total = int(round(seconds))
    return f"{total // 60:02d}:{total % 60:02d}"


def get_video_duration_seconds(video_name, video_path):
    """Read duration with OpenCV; fall back to known assignment metadata."""
    try:
        import cv2

        cap = cv2.VideoCapture(str(video_path))
        if cap.isOpened():
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
            cap.release()
            if fps and frame_count:
                return frame_count / fps
    except Exception as exc:
        print(f"OpenCV duration read failed for {video_name}; using fallback. Reason: {exc}")

    return KNOWN_DURATIONS[video_name]


def make_chunk_ranges(duration_s, max_chunk_s=MAX_CHUNK_SECONDS, overlap_s=CHUNK_OVERLAP_SECONDS):
    """Return overlapping chunk ranges as (start_s, end_s) pairs."""
    if duration_s <= max_chunk_s:
        return [(0.0, duration_s)]

    chunks = []
    step = max_chunk_s - overlap_s
    start = 0.0

    while start < duration_s:
        end = min(start + max_chunk_s, duration_s)
        chunks.append((start, end))

        if end >= duration_s:
            break

        start += step

    return chunks


video_chunk_plan = {}

for video_name, video_path in VIDEO_PATHS.items():
    if not video_path.exists():
        raise FileNotFoundError(f"Missing video file: {video_path}")

    duration_s = get_video_duration_seconds(video_name, video_path)
    chunks = make_chunk_ranges(duration_s)

    video_chunk_plan[video_name] = {
        "video_path": str(video_path),
        "duration_s": duration_s,
        "duration": seconds_to_mmss(duration_s),
        "chunks": [
            {
                "chunk_id": chunk_id,
                "start_s": start_s,
                "end_s": end_s,
                "start": seconds_to_mmss(start_s),
                "end": seconds_to_mmss(end_s),
            }
            for chunk_id, (start_s, end_s) in enumerate(chunks, start=1)
        ],
    }


print("Video chunk plan:")
for video_name, plan in video_chunk_plan.items():
    print(f"\n{video_name}: duration {plan['duration']} ({plan['duration_s']:.1f}s), {len(plan['chunks'])} chunk(s)")
    for chunk in plan["chunks"]:
        print(f"  chunk {chunk['chunk_id']}: {chunk['start']} - {chunk['end']}")


assert len(video_chunk_plan["video_21"]["chunks"]) > 1, "video_21 should be split into multiple chunks."
assert len(video_chunk_plan["video_24"]["chunks"]) == 1, "video_24 should fit as one chunk."
print("\nStep 6.3 check passed: video durations and chunk ranges are ready.")

## Step 6.4 - Check Retrieval Framework and Model Setup

We will use **Lighthouse** because it provides a common inference API for several pretrained moment-retrieval models.

For the assignment requirement of using two models, we will start with:

1. **Moment-DETR**
2. **CG-DETR**

For the feature experiment, we will start with:

1. **CLIP** features - practical baseline, especially on CPU
2. **CLIP + SlowFast** features - optional second feature condition if compute allows

According to the Lighthouse documentation, the setup is:

```bash
pip install torch torchvision torchaudio
pip install "git+https://github.com/line/lighthouse.git"
```

This cell only checks whether the environment is ready. It does **not** run inference yet.

In [ ]:
import importlib.util
import sys
from pathlib import Path

WEIGHTS_DIR = Path("weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

RETRIEVAL_MODELS = [
    {
        "model_key": "moment_detr",
        "display_name": "Moment-DETR",
        "predictor_class": "MomentDETRPredictor",
    },
    {
        "model_key": "cg_detr",
        "display_name": "CG-DETR",
        "predictor_class": "CGDETRPredictor",
    },
]

# First feature condition. This is the practical baseline recommended for CPU runs.
FEATURES_TO_TEST = ["clip"]

# Optional second feature condition. We will only run this if the environment can handle it.
OPTIONAL_FEATURES_TO_TEST = ["clip_slowfast"]


def package_available(package_name):
    return importlib.util.find_spec(package_name) is not None


torch_available = package_available("torch")
lighthouse_available = package_available("lighthouse")

if torch_available:
    import torch

    if torch.cuda.is_available():
        RETRIEVAL_DEVICE = "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        RETRIEVAL_DEVICE = "mps"
    else:
        RETRIEVAL_DEVICE = "cpu"
else:
    RETRIEVAL_DEVICE = "cpu"


available_predictor_classes = {}
model_import_error = None
if lighthouse_available:
    try:
        # Compatibility shim: some dependencies try `import numpy.char`.
        import numpy.core.defchararray as numpy_char
        sys.modules.setdefault("numpy.char", numpy_char)

        import lighthouse.models as lighthouse_models

        for model_cfg in RETRIEVAL_MODELS:
            class_name = model_cfg["predictor_class"]
            available_predictor_classes[class_name] = hasattr(lighthouse_models, class_name)
    except Exception as exc:
        model_import_error = str(exc)
        available_predictor_classes = {model_cfg["predictor_class"]: False for model_cfg in RETRIEVAL_MODELS}
else:
    available_predictor_classes = {model_cfg["predictor_class"]: False for model_cfg in RETRIEVAL_MODELS}


checkpoint_plan = []
for model_cfg in RETRIEVAL_MODELS:
    for feature_name in FEATURES_TO_TEST:
        checkpoint_plan.append({
            "model": model_cfg["model_key"],
            "feature": feature_name,
            "path": WEIGHTS_DIR / f"{feature_name}_{model_cfg['model_key']}_qvhighlight.ckpt",
        })
retrieval_setup = {
    "device": RETRIEVAL_DEVICE,
    "models": RETRIEVAL_MODELS,
    "features": FEATURES_TO_TEST,
    "optional_features": OPTIONAL_FEATURES_TO_TEST,
    "checkpoint_plan": checkpoint_plan,
}


missing_checkpoints = [item for item in checkpoint_plan if not item["path"].exists()]

models_ready = all(available_predictor_classes.values())

print("Step 6.4")
print(f"Models ready: {models_ready}")
print(f"Checkpoints found: {len(checkpoint_plan) - len(missing_checkpoints)}/{len(checkpoint_plan)}")

if missing_checkpoints:
    for item in missing_checkpoints:
        print(f"Missing: {item['path']}")

if torch_available and lighthouse_available and models_ready:
    print("Step 6.4 passed")
else:
    print("Step 6.4 not ready")

## Step 6.5 - Run One Moment-DETR Test Query

This step checks that the first pretrained model actually works before we run it over all events.

We will:

1. download/check the **Moment-DETR + CLIP** checkpoint,
2. load `MomentDETRPredictor`,
3. run one query from `video_24`,
4. print only the predicted timestamp and confidence score.

Success check: the model should return one predicted time window.

In [ ]:
import contextlib
import os
import sys
import urllib.request

assert "query_variants" in globals(), "Run Step 6.2 first."
assert "video_chunk_plan" in globals(), "Run Step 6.3 first."
assert "retrieval_setup" in globals(), "Run Step 6.4 first."

MOMENT_DETR_CKPT = WEIGHTS_DIR / "clip_moment_detr_qvhighlight.ckpt"
MOMENT_DETR_SAFE_CKPT = WEIGHTS_DIR / "clip_moment_detr_qvhighlight_safe.ckpt"
MOMENT_DETR_URL = "https://zenodo.org/records/13363606/files/clip_moment_detr_qvhighlight.ckpt"


def download_file_if_missing(url, output_path):
    output_path.parent.mkdir(exist_ok=True)
    if output_path.exists():
        return False

    temp_path = output_path.with_suffix(output_path.suffix + ".part")
    urllib.request.urlretrieve(url, temp_path)
    temp_path.replace(output_path)
    return True


# Compatibility shim for Lighthouse imports in some Windows/NumPy combinations.
import numpy.core.defchararray as numpy_char
sys.modules.setdefault("numpy.char", numpy_char)

from lighthouse.models import MomentDETRPredictor
from easydict import EasyDict
import torch


def create_safe_lighthouse_checkpoint(source_path, output_path):
    """Convert the official checkpoint's EasyDict config into a plain dict.

    PyTorch's safe loader cannot unpickle EasyDict directly. This conversion is
    done once from the official Lighthouse checkpoint, then later loads use the
    converted plain-dict checkpoint.
    """
    if output_path.exists():
        return False

    original = torch.load(source_path, map_location="cpu", weights_only=False)
    safe_checkpoint = {
        "model": original["model"],
        "opt": dict(original["opt"]),
    }
    torch.save(safe_checkpoint, output_path)
    return True


downloaded = download_file_if_missing(MOMENT_DETR_URL, MOMENT_DETR_CKPT)
converted = create_safe_lighthouse_checkpoint(MOMENT_DETR_CKPT, MOMENT_DETR_SAFE_CKPT)

# Use video_23 for the smoke test because it has a clearer narrative than
# video_24, which is a fast compilation of unrelated cuts.
TEST_VIDEO_NAME = "video_23"
TEST_EVENT_ID = 5

test_chunk = video_chunk_plan[TEST_VIDEO_NAME]["chunks"][0]
test_video = video_chunk_plan[TEST_VIDEO_NAME]["video_path"]
test_event = query_variants[TEST_VIDEO_NAME][TEST_EVENT_ID - 1]
test_query = test_event["queries"][0]


def make_temp_video_chunk(video_path, chunk, output_dir=Path("outputs/moment_retrieval_chunks")):
    """Create a physical video chunk for retrieval if it does not exist yet."""
    output_dir.mkdir(parents=True, exist_ok=True)
    chunk_path = output_dir / f"{Path(video_path).stem}_chunk_{chunk['chunk_id']:02d}.mp4"
    if chunk_path.exists():
        return chunk_path

    import subprocess
    duration = chunk["end_s"] - chunk["start_s"]
    command = [
        "ffmpeg",
        "-y",
        "-ss", str(chunk["start_s"]),
        "-t", str(duration),
        "-i", str(video_path),
        "-c", "copy",
        str(chunk_path),
    ]
    subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return chunk_path
CLIP_CACHE_DIR = WEIGHTS_DIR / "clip_cache"
CLIP_CACHE_DIR.mkdir(exist_ok=True)

# Make conda-installed ffmpeg/ffprobe visible to ffmpeg-python.
CONDA_FFMPEG_BIN = Path(sys.prefix) / "Library" / "bin"
if CONDA_FFMPEG_BIN.exists():
    os.environ["PATH"] = str(CONDA_FFMPEG_BIN) + os.pathsep + os.environ.get("PATH", "")

# Lighthouse expects attribute-style config access. The converted checkpoint
# stores the config safely as a dict, so we turn it back into EasyDict after
# torch.load has completed.
_original_torch_load = torch.load


def _torch_load_safe_lighthouse(*args, **kwargs):
    loaded = _original_torch_load(*args, **kwargs)
    if args and Path(args[0]).resolve() == MOMENT_DETR_SAFE_CKPT.resolve():
        loaded["opt"] = EasyDict(loaded["opt"])
    return loaded


# Keep CLIP downloads inside the project instead of the user home cache.
import clip
_original_clip_load = clip.load


def _clip_load_project_cache(*args, **kwargs):
    kwargs.setdefault("download_root", str(CLIP_CACHE_DIR))
    return _original_clip_load(*args, **kwargs)

# Suppress library progress/log chatter so the notebook output stays readable.
try:
    torch.load = _torch_load_safe_lighthouse
    clip.load = _clip_load_project_cache
    with open("nul", "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            moment_detr_model = MomentDETRPredictor(
                str(MOMENT_DETR_SAFE_CKPT),
                device=retrieval_setup["device"],
                feature_name="clip",
            )
            test_chunk_path = make_temp_video_chunk(test_video, test_chunk)
            encoded_video = moment_detr_model.encode_video(str(test_chunk_path))
            prediction = moment_detr_model.predict(test_query, encoded_video)
finally:
    torch.load = _original_torch_load
    clip.load = _original_clip_load

best_window = prediction["pred_relevant_windows"][0]
chunk_start_s, chunk_end_s, score = best_window
start_s = test_chunk["start_s"] + chunk_start_s
end_s = test_chunk["start_s"] + chunk_end_s

print("Step 6.5")
print(f"Checkpoint ready: {MOMENT_DETR_SAFE_CKPT.exists()}")
print(f"Video: {TEST_VIDEO_NAME}")
print(f"Chunk: {test_chunk['start']} - {test_chunk['end']}")
print(f"Query: {test_query}")
print(f"Prediction: {seconds_to_mmss(start_s)} - {seconds_to_mmss(end_s)}")
print(f"Score: {score:.4f}")
print("Step 6.5 passed")

## Step 6.6 - Run One CG-DETR Test Query

The Canvas demo uses `CGDETRPredictor`, so this step mirrors that demo more directly.

We use the same video chunk and query as Step 6.5 so the two smoke tests are comparable:

- **Step 6.5:** Moment-DETR + CLIP
- **Step 6.6:** CG-DETR + CLIP

Success check: CG-DETR should return one predicted time window and score.

In [ ]:
import contextlib
import os
import sys
import urllib.request

assert "query_variants" in globals(), "Run Step 6.2 first."
assert "video_chunk_plan" in globals(), "Run Step 6.3 first."
assert "retrieval_setup" in globals(), "Run Step 6.4 first."

CG_DETR_CKPT = WEIGHTS_DIR / "clip_cg_detr_qvhighlight.ckpt"
CG_DETR_SAFE_CKPT = WEIGHTS_DIR / "clip_cg_detr_qvhighlight_safe.ckpt"
CG_DETR_URL = "https://zenodo.org/records/13363606/files/clip_cg_detr_qvhighlight.ckpt"


def download_file_if_missing(url, output_path):
    output_path.parent.mkdir(exist_ok=True)
    if output_path.exists():
        return False

    temp_path = output_path.with_suffix(output_path.suffix + ".part")
    urllib.request.urlretrieve(url, temp_path)
    temp_path.replace(output_path)
    return True


# Compatibility shim for Lighthouse imports in some Windows/NumPy combinations.
import numpy.core.defchararray as numpy_char
sys.modules.setdefault("numpy.char", numpy_char)

from lighthouse.models import CGDETRPredictor
from easydict import EasyDict
import torch


def create_safe_lighthouse_checkpoint(source_path, output_path):
    """Convert the official checkpoint's EasyDict config into a plain dict."""
    if output_path.exists():
        return False

    original = torch.load(source_path, map_location="cpu", weights_only=False)
    safe_checkpoint = {
        "model": original["model"],
        "opt": dict(original["opt"]),
    }
    torch.save(safe_checkpoint, output_path)
    return True


downloaded = download_file_if_missing(CG_DETR_URL, CG_DETR_CKPT)
converted = create_safe_lighthouse_checkpoint(CG_DETR_CKPT, CG_DETR_SAFE_CKPT)

TEST_VIDEO_NAME = "video_23"
TEST_EVENT_ID = 5

test_chunk = video_chunk_plan[TEST_VIDEO_NAME]["chunks"][0]
test_video = video_chunk_plan[TEST_VIDEO_NAME]["video_path"]
test_event = query_variants[TEST_VIDEO_NAME][TEST_EVENT_ID - 1]
test_query = test_event["queries"][0]


def make_temp_video_chunk(video_path, chunk, output_dir=Path("outputs/moment_retrieval_chunks")):
    """Create a physical video chunk for retrieval if it does not exist yet."""
    output_dir.mkdir(parents=True, exist_ok=True)
    chunk_path = output_dir / f"{Path(video_path).stem}_chunk_{chunk['chunk_id']:02d}.mp4"
    if chunk_path.exists():
        return chunk_path

    import subprocess
    duration = chunk["end_s"] - chunk["start_s"]
    command = [
        "ffmpeg",
        "-y",
        "-ss", str(chunk["start_s"]),
        "-t", str(duration),
        "-i", str(video_path),
        "-c", "copy",
        str(chunk_path),
    ]
    subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return chunk_path


CLIP_CACHE_DIR = WEIGHTS_DIR / "clip_cache"
CLIP_CACHE_DIR.mkdir(exist_ok=True)

CONDA_FFMPEG_BIN = Path(sys.prefix) / "Library" / "bin"
if CONDA_FFMPEG_BIN.exists():
    os.environ["PATH"] = str(CONDA_FFMPEG_BIN) + os.pathsep + os.environ.get("PATH", "")

_original_torch_load = torch.load


def _torch_load_safe_lighthouse(*args, **kwargs):
    loaded = _original_torch_load(*args, **kwargs)
    if args and Path(args[0]).resolve() == CG_DETR_SAFE_CKPT.resolve():
        loaded["opt"] = EasyDict(loaded["opt"])
    return loaded


import clip
_original_clip_load = clip.load


def _clip_load_project_cache(*args, **kwargs):
    kwargs.setdefault("download_root", str(CLIP_CACHE_DIR))
    return _original_clip_load(*args, **kwargs)


try:
    torch.load = _torch_load_safe_lighthouse
    clip.load = _clip_load_project_cache
    with open("nul", "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            cg_detr_model = CGDETRPredictor(
                str(CG_DETR_SAFE_CKPT),
                device=retrieval_setup["device"],
                feature_name="clip",
            )
            test_chunk_path = make_temp_video_chunk(test_video, test_chunk)
            encoded_video = cg_detr_model.encode_video(str(test_chunk_path))
            prediction = cg_detr_model.predict(test_query, encoded_video)
finally:
    torch.load = _original_torch_load
    clip.load = _original_clip_load

best_window = prediction["pred_relevant_windows"][0]
chunk_start_s, chunk_end_s, score = best_window
start_s = test_chunk["start_s"] + chunk_start_s
end_s = test_chunk["start_s"] + chunk_end_s

print("Step 6.6")
print(f"Checkpoint ready: {CG_DETR_SAFE_CKPT.exists()}")
print(f"Video: {TEST_VIDEO_NAME}")
print(f"Chunk: {test_chunk['start']} - {test_chunk['end']}")
print(f"Query: {test_query}")
print(f"Prediction: {seconds_to_mmss(start_s)} - {seconds_to_mmss(end_s)}")
print(f"Score: {score:.4f}")
print("Step 6.6 passed")